# Forward 2D Poisson — PIFT on [0,1]²

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmhobbs96/pift-od-il-inverse-problems/blob/main/examples/02_forward_poisson_2d.ipynb)

This notebook extends the 1D PIFT framework to the 2D Poisson equation
$-(\partial_{xx} + \partial_{yy})\phi = f$ on the unit square $[0,1]^2$ with zero Dirichlet
boundary conditions on all four sides.

The field is parameterized with a **SineBasis2D** tensor-product expansion
$$\phi(x,y;\theta) = \sum_{j=1}^K \sum_{k=1}^K \theta_{jk} \sin(j\pi x)\sin(k\pi y),$$
which exactly satisfies all BCs. With $K=6$ side-modes we get $K^2=36$ parameters.
The SGLD sampler, physics energy, and likelihood follow the same structure as notebook 01,
extended to 2D via the `run_phase_e_2d_poisson` pipeline.

In [ ]:
%pip install -q git+https://github.com/cmhobbs96/pift-od-il-inverse-problems.git
%pip install -q tqdm

import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print('JAX backend:', jax.default_backend(), '| devices:', jax.devices())

from pipelines.phase_e_2d_poisson import run_phase_e_2d_poisson

def show_fig(fig, dpi=120):
    import tempfile
    from IPython.display import Image, display
    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
        fig.savefig(f.name, dpi=dpi, bbox_inches='tight')
        plt.close(fig)
        display(Image(f.name))

## Configuration

In [ ]:
CONFIG = {
    'seed':      7,
    'n_modes':   6,      # SineBasis2D side -> 36 params total                [3, 12]
    'n_obs':     80,     # 2D observations scattered on [0,1]^2               [10, 400]
    'noise_std': 0.04,   # observation noise sigma                            [1e-4, 0.5]
    'beta':      5.0,    # physics trust beta                                 [0.01, 100]
    'n_steps':   5000,   # SGLD iterations                                    [500, 50000]
    'burn_in':   1000,   # warm-up discarded                                  [100, n_steps/2]
    'n_quad':    256,    # 2D quadrature points per step                      [64, 2048]
    'n_grid':    51,     # grid points per axis (51x51 = 2601 total)          [21, 201]
}

## Run 2D PIFT

The pipeline handles random truth generation, FD reference (on a structured grid),
observation sampling, SGLD, and chain post-processing. It returns a results dictionary
containing the full chain, grid arrays, and computed metrics.

In [ ]:
results = run_phase_e_2d_poisson(
    config=CONFIG,
    device_preference=jax.default_backend(),
    save_outputs=False,
)

print('Status:    ', results['status'])
print(f'Runtime:   {results["runtime_s"]:.1f} s')
print(f'L2 error:  {results["l2_error"]:.4f}')
print(f'Max error: {results["max_error"]:.4f}')
print(f'Samples:   {results["n_samples"]}')
print(f'Params:    {results["n_params"]} (K={CONFIG["n_modes"]} -> K^2={CONFIG["n_modes"]**2})')

## Results

We visualize the truth, posterior mean, posterior standard deviation, and pointwise
absolute error as a four-panel figure, followed by the Hamiltonian energy trace.

In [ ]:
ng = CONFIG['n_grid']

phi_truth_2d = results['phi_truth'].reshape(ng, ng)
phi_mean_2d  = results['phi_mean'].reshape(ng, ng)
phi_std_2d   = results['phi_std'].reshape(ng, ng)
phi_err_2d   = np.abs(phi_mean_2d - phi_truth_2d)

titles  = ['Truth $\\phi^\\star$', 'Posterior Mean', 'Posterior Std Dev', 'Abs Error']
arrays  = [phi_truth_2d, phi_mean_2d, phi_std_2d, phi_err_2d]
cmaps   = ['viridis', 'viridis', 'plasma', 'hot_r']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, arr, title, cmap in zip(axes, arrays, titles, cmaps):
    im = ax.imshow(
        arr, origin='lower', extent=[0, 1, 0, 1],
        cmap=cmap, aspect='equal'
    )
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(
    f'2D Poisson PIFT  |  $\\beta$={CONFIG["beta"]}, K={CONFIG["n_modes"]}  '
    f'|  L2={results["l2_error"]:.4f}',
    fontsize=13
)
fig.tight_layout()
show_fig(fig)

# --- Energy trace ---
hamiltonians = np.array(results['hamiltonians'])
fig2, ax2 = plt.subplots(figsize=(8, 3))
ax2.plot(hamiltonians[CONFIG['burn_in']:][::5], lw=0.7, color='purple', alpha=0.8)
ax2.set_xlabel('Post-burn-in step / 5')
ax2.set_ylabel('Hamiltonian')
ax2.set_title('2D PIFT — Hamiltonian Trace')
fig2.tight_layout()
show_fig(fig2)